# 贝叶斯岭回归：自带不确定性估计的回归
> 难度标签：初级 | 预计时长：15-25分钟 | 前置知识：无学习经验


> 所属模块：02_监督学习/回归 | 源文件：贝叶斯岭回归.py | 核心功能：贝叶斯推断、预测不确定性估计、自动正则化参数

## 概述

贝叶斯岭回归是岭回归的贝叶斯版本。不把权重看作固定未知数，而是视为**随机变量**——先验是高斯分布，通过贝叶斯推断获得后验分布。这带来两个独特优势：

1. **自动学习正则化参数**：不需要手动调 alpha
2. **预测不确定性估计**：predict 可以返回均值和标准差

脚本对比了线性回归、岭回归和贝叶斯岭回归，并展示了预测置信区间。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import BayesianRidge, LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score
# --- 导入库 ---
from sklearn.preprocessing import StandardScaler

## 数学原理

### 1. 贝叶斯线性回归框架

**代码对应**：`BayesianRidge().fit(X_train, y_train)` 使用贝叶斯推断估计参数。

贝叶斯线性回归将权重 $\mathbf{w}$ 视为随机变量，通过贝叶斯定理更新其分布：

**似然**（数据模型）：

$$P(\mathbf{y}|\mathbf{X}, \mathbf{w}, \lambda) = \mathcal{N}(\mathbf{y}|\mathbf{X}\mathbf{w}, \lambda^{-1}\mathbf{I})$$

其中 $\lambda = 1/\sigma^2$ 为噪声精度（inverse variance）。

**先验**（权重的先验信念）：

$$P(\mathbf{w}|\alpha) = \mathcal{N}(\mathbf{w}|\mathbf{0}, \alpha^{-1}\mathbf{I})$$

其中 $\alpha = 1/\tau^2$ 为权重精度。这正是 Ridge 回归的 L2 正则化的贝叶斯解释。

### 2. 后验分布推导

由贝叶斯定理：

$$P(\mathbf{w}|\mathbf{y}, \mathbf{X}, \alpha, \lambda) \propto P(\mathbf{y}|\mathbf{X}, \mathbf{w}, \lambda) \cdot P(\mathbf{w}|\alpha)$$

两个高斯分布的乘积仍是高斯分布。后验为：

$$P(\mathbf{w}|\mathbf{y}) = \mathcal{N}(\mathbf{w}|\boldsymbol{\mu}_N, \boldsymbol{\Sigma}_N)$$

其中：

$$\boldsymbol{\Sigma}_N = (\alpha\mathbf{I} + \lambda\mathbf{X}^T\mathbf{X})^{-1}$$

$$\boldsymbol{\mu}_N = \lambda\boldsymbol{\Sigma}_N\mathbf{X}^T\mathbf{y}$$

**代码对应**：`br.coef_` 返回后验均值 $\boldsymbol{\mu}_N$，即系数的最佳估计。后验协方差 $\boldsymbol{\Sigma}_N$ 给出系数的不确定性。

### 3. 超参数的自动学习（Type-II 最大似然）

**代码对应**：sklearn 的 `BayesianRidge` 自动学习 `alpha_`（权重精度）和 `lambda_`（噪声精度）。

$\alpha$ 和 $\lambda$ 通过**证据近似**（Evidence Approximation）或**Type-II 最大似然**估计：

$$\ln P(\mathbf{y}|\alpha, \lambda) = \frac{p}{2}\ln\alpha + \frac{n}{2}\ln\lambda - E(\boldsymbol{\mu}_N) - \frac{1}{2}\ln|\mathbf{A}| - \frac{n}{2}\ln(2\pi)$$

其中 $\mathbf{A} = \alpha\mathbf{I} + \lambda\mathbf{X}^T\mathbf{X}$，$E(\boldsymbol{\mu}_N) = \frac{\lambda}{2}\|\mathbf{y} - \mathbf{X}\boldsymbol{\mu}_N\|^2 + \frac{\alpha}{2}\|\boldsymbol{\mu}_N\|^2$。

通过 EM 算法迭代更新：

$$\alpha^{(t+1)} = \frac{p}{\|\boldsymbol{\mu}_N\|^2 + \text{tr}(\boldsymbol{\Sigma}_N)}, \quad \lambda^{(t+1)} = \frac{n}{\|\mathbf{y} - \mathbf{X}\boldsymbol{\mu}_N\|^2 + \text{tr}(\mathbf{X}^T\mathbf{X}\boldsymbol{\Sigma}_N)}$$

**关键优势**：无需手动调正则化参数——数据自动决定收缩强度。

### 4. 预测分布与不确定性估计

**代码对应**：`br.predict(X_test, return_std=True)` 返回预测均值和标准差。

给定新样本 $\mathbf{x}_*$，预测分布为：

$$P(y_*|\mathbf{x}_*, \mathbf{y}) = \mathcal{N}(y_*|\boldsymbol{\mu}_N^T\mathbf{x}_*, \sigma_*^2)$$

其中预测方差为：

$$\sigma_*^2 = \frac{1}{\lambda} + \mathbf{x}_*^T\boldsymbol{\Sigma}_N\mathbf{x}_*$$

第一项 $1/\lambda$ 是**数据噪声**（不可约），第二项 $\mathbf{x}_*^T\boldsymbol{\Sigma}_N\mathbf{x}_*$ 是**参数不确定性**（可通过更多数据减小）。

**代码对应**：代码中用 `y_pred_mean - 1.96 * y_pred_std` 和 `y_pred_mean + 1.96 * y_pred_std` 构造 95% 置信区间：

$$\text{95\% CI} = \hat{y}_* \pm 1.96 \cdot \sigma_*$$

### 5. 与普通岭回归的关系

| 特性 | Ridge | Bayesian Ridge |
|------|-------|----------------|
| 正则化参数 $\alpha$ | 手动设置或交叉验证 | 自动从数据学习 |
| 预测输出 | 点估计 $\hat{y}$ | 均值 + 标准差 |
| 不确定性估计 | 无 | 有（预测分布） |
| 贝叶斯先验 | $\mathbf{w} \sim \mathcal{N}(0, \alpha^{-1}\mathbf{I})$，$\alpha$ 固定 | $\mathbf{w} \sim \mathcal{N}(0, \alpha^{-1}\mathbf{I})$，$\alpha$ 有 Gamma 先验 |

### 6. 先验超参数的设置

**代码对应**：`alpha_1, alpha_2, lambda_1, lambda_2` 控制 Gamma 先验的形状。

$\alpha$ 和 $\lambda$ 本身也有先验：

$$\alpha \sim \text{Gamma}(\alpha_1, \alpha_2), \quad \lambda \sim \text{Gamma}(\lambda_1, \lambda_2)$$

Gamma 先验的均值为 $\alpha_1/\alpha_2$，方差为 $\alpha_1/\alpha_2^2$。默认值 $\alpha_1 = \alpha_2 = \lambda_1 = \lambda_2 = 10^{-6}$ 对应非常弱的先验（让数据主导）。

### 1. 生成数据

生成合成数据集，用于演示算法的核心行为。

In [ ]:
np.random.seed(42)
X, y = make_regression(n_samples=200, n_features=5, n_informative=3, noise=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
# --- 继续 ---
X_test = scaler.transform(X_test)

### 2. 三种回归对比

在回归任务上拟合模型，观察预测值与真实值的偏差。

In [ ]:
lr = LinearRegression().fit(X_train, y_train)
ridge = Ridge(alpha=1.0).fit(X_train, y_train)
br = BayesianRidge().fit(X_train, y_train)

print("=== 线性回归 vs 岭回归 vs 贝叶斯岭回归 ===")
print(f"{'方法':>20} {'测试 R²':>10}")
print(f"{'线性回归':>20} {lr.score(X_test, y_test):>10.4f}")
print(f"{'岭回归':>20} {ridge.score(X_test, y_test):>10.4f}")
print(f"{'贝叶斯岭回归':>20} {br.score(X_test, y_test):>10.4f}")

### 3. 不确定性估计

运行 3. 不确定性估计 的代码，观察算法在该环节的行为。

In [ ]:
# predict 返回均值和标准差，标准差反映预测不确定性
y_pred_mean, y_pred_std = br.predict(X_test, return_std=True)
print(f"\n=== 预测不确定性 ===")
print(f"{'样本':>6} {'真实值':>10} {'预测均值':>10} {'预测标准差':>12} {'95%区间':>20}")
for i in range(10):
    lo = y_pred_mean[i] - 1.96 * y_pred_std[i]
# --- 赋值/配置 ---
    hi = y_pred_mean[i] + 1.96 * y_pred_std[i]
    print(f"  {i+1:>4} {y_test[i]:>10.2f} {y_pred_mean[i]:>10.2f} "
          f"{y_pred_std[i]:>12.4f} [{lo:>8.2f}, {hi:>8.2f}]")

### 4. 贝叶斯岭 vs 普通岭回归

在回归任务上拟合模型，观察预测值与真实值的偏差。

In [ ]:
print("\n=== 贝叶斯岭 vs 岭回归 ===")
print(f"贝叶斯岭 α (精度): {br.alpha_:.6f}")
print(f"贝叶斯岭 λ (噪声): {br.lambda_:.6f}")
print(f"贝叶斯岭 系数: {br.coef_.round(4)}")
print(f"岭回归   系数: {ridge.coef_.round(4)}")

### 5. 超参数调优

探索关键超参数对模型行为的影响。

In [ ]:
print("\n=== 关键超参数 ===")
print("BayesianRidge 的 alpha_1/alpha_2/lambda_1/lambda_2 是先验分布的超参数")
print(f"  alpha_1={br.alpha_1}, alpha_2={br.alpha_2} (控制 α 的 Gamma 先验)")
print(f"  lambda_1={br.lambda_1}, lambda_2={br.lambda_2} (控制 λ 的 Gamma 先验)")

# 手动调参
for alpha_1 in [1e-6, 1e-4, 1e-2]:
    br_t = BayesianRidge(alpha_1=alpha_1, max_iter=300)
    br_t.fit(X_train, y_train)
    print(f"  alpha_1={alpha_1}: R²={br_t.score(X_test, y_test):.4f}, "
          f"alpha_={br_t.alpha_:.4f}, lambda_={br_t.lambda_:.4f}")

### 6. 概率预测（多输出）

使用训练好的模型进行预测，观察输出结果。

In [ ]:
# BayesianRidge 不支持原生多输出，需用 MultiOutputRegressor 包装
from sklearn.multioutput import MultiOutputRegressor

print("\n=== 多目标回归（MultiOutputRegressor + BayesianRidge）===")
y_multi = np.column_stack([y, y * 0.5 + np.random.randn(len(y)) * 5])
br_multi = MultiOutputRegressor(BayesianRidge())
br_multi.fit(X_train, y_multi[:len(X_train)])
y_pred_multi = br_multi.predict(X_test)
# --- 输出结果 ---
print(f"多输出预测形状: {y_pred_multi.shape}")
print(f"前 5 个样本预测值:\n{y_pred_multi[:5].round(2)}")

print("\n=== 贝叶斯岭回归要点 ===")
print("- 将权重视为随机变量，用贝叶斯推断估计后验分布")
print("- 自动学习正则化参数 alpha（权重精度）和 lambda（噪声精度）")
print("- 提供预测不确定性估计（标准差），适合需要置信区间的场景")
print("- 系数被 收缩 向 0（类似岭回归），但收缩强度由数据自适应决定")
# --- 输出结果 ---
print("- 适合：小样本、需要不确定性估计、不想手动调正则化参数")

## 关键代码解释

### 不确定性估计

```python
y_pred_mean, y_pred_std = br.predict(X_test, return_std=True)
# 95% 置信区间 = 预测均值 ± 1.96 × 标准差
```

这是贝叶斯回归最独特的能力——不仅给出预测值，还告诉你"模型对这个预测有多自信"。不确定性大的区域意味着模型缺乏足够数据做出可靠判断。

### 自动正则化参数

```python
print(f"alpha (权重精度): {br.alpha_:.6f}")
print(f"lambda (噪声精度): {br.lambda_:.6f}")
```

alpha 和 lambda 是模型自动学习的超参数，不需要手动调参。alpha 越大，正则化越强（权重越小）。

## 使用示例

```python
from sklearn.linear_model import BayesianRidge
br = BayesianRidge()
br.fit(X_train, y_train)
y_mean, y_std = br.predict(X_test, return_std=True)
```

## 注意事项

1. **计算比岭回归慢**：需要迭代优化
2. **正态假设**：假设噪声和权重先验都是高斯的
3. **不确定性不是校准的**：预测标准差可能偏大或偏小
4. **不支持多输出**：需要用 MultiOutputRegressor 包装

## 总结与延伸

以上代码展示了 **贝叶斯岭回归** 的完整流程。

**解读要点**：
- 关注 **R² 分数** 和 **MSE/MAE** 等回归指标
- R² 越接近 1 说明拟合越好
- 观察预测值 vs 真实值的散点图是否沿对角线分布

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **ARD（自动相关性确定）回归**：给每个特征独立的精度参数，自动识别无关特征
- **高斯过程回归**：非参数贝叶斯回归，更灵活但计算更贵
- **变分推断**：大数据集上的近似贝叶斯推断方法
- **不确定性估计的应用**：主动学习、异常检测、风险评估
